In [1]:
import os
import uuid

from dotenv import load_dotenv

import xarray as xr
import xesmf as xe # type: ignore[ty:unresolved-import]
import datetime

In [10]:
load_dotenv()

OUT_ZARR = "/home/emily_zuetell/projects/poreallas/data/parsed/26_era5_daily_running.zarr"
TARGET_REGRID_URI = "/home/emily_zuetell/projects/poreallas/data/s51_hcm.nc"
UID = str(uuid.uuid4())
START_TIME = datetime.datetime.now(datetime.UTC).isoformat()

In [4]:
def open_regrid_target(uri: str) -> xr.Dataset:
    """Open/clean a dataset to use as a regridding target"""
    # Using the S51 seasonal monthly seasonal hindcast ensemble mean from copernicus as the target grid for our regrid...
    # Selecting so only have coords for latitude and longitude for regridding.
    target = xr.open_dataset(uri).isel(
        {"forecast_reference_time": 0, "forecastMonth": 0}, drop=True
    )
    return target

def avg_era5(base_file):
    """Average the daily min and max datasets to get daily mean"""
    # Open daily min and max era5 datasets
    era5_min = xr.open_dataset(f"{base_file}_min.nc")
    era5_max = xr.open_dataset(f"{base_file}_max.nc")
    # Daily average
    era5 = (era5_min + era5_max)/2
    # Rename the time dimension
    era5 = era5.rename({'valid_time': 'time'})
    return era5

In [5]:
regrid_target = open_regrid_target(TARGET_REGRID_URI)

In [7]:
era5 = avg_era5("/home/emily_zuetell/projects/poreallas/data/raw/26_era5_daily")

In [8]:
# Cannot have leap years in QDM bias adjustment so convert to a no-leapyear calendar.
era5 = era5.convert_calendar("noleap", dim="time")
# Regrid the era5 dataset to the target grid (ECMWF)
regridder = xe.Regridder(era5, regrid_target, method="bilinear", periodic=True)
era5_regrid = regridder(era5)
era5_regrid.attrs |= era5.attrs

In [11]:
# Metadata on units is required later in the workflow.
era5_regrid["t2m"].attrs["units"] = "K"

# Add additional general metadata.
era5_regrid.attrs |= {
    "poreallas_created_at": START_TIME,
    "poreallas_uid": UID,
    "poreallas_description": "Parsed ERA5 (running 2026) climate fields",
}
era5_regrid["t2m"].attrs |= {
    "poreallas_created_at": START_TIME,
    "poreallas_uid": UID,
    "poreallas_description": "Parsed ERA5 (running 2026) tas field",
}

# All of time needs to be in a single chunk for QDM bias adjustment.
# This generally gets you ~110 MiB chunks.
era5_regrid = era5_regrid.chunk({"time": -1, "latitude": 30, "longitude": 60})

era5_regrid.to_zarr(OUT_ZARR, consolidated=True)

/home/emily_zuetell/miniforge3/envs/esmf-env/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
